# Inference cost on a Colab T4 (second platform)

Re-measures the step-8 configurations of `coco_ViT-L-14_lora8_lr1e-3_s0` (epoch_02) on an NVIDIA T4 with the same
code and design as on the Mac (`scripts/10_measure_variants.py`): 12 configurations, captions and answers at batch 1,
500 test images and 500 questions, one encoder window per precision and one decoding window per configuration and
task, idle once per repetition, 3 repetitions. On the T4, bitsandbytes has CUDA kernels for INT8 and NF4, which the
Mac lacks for INT8.

Energy: NVML's board energy counter, **GPU only** (Colab gives no CPU or DRAM counters), so the T4 frontier is its
own and is never put on one axis with the Mac's (SoC + DRAM).

Before you start: upload `greenvl_t4_bundle.zip` to `My Drive/greenvl_t4/`. Then Runtime → Change runtime type →
**T4 GPU**, and run the cells in order. Keep this tab open (the free tier stops when the tab closes or after long
inactivity). Results are written to `My Drive/greenvl_t4/results/` after every window, so a disconnect loses at most
one window: reconnect, run cells 1-4 again, then cell 7, and it continues where it stopped.

### 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,power.limit,memory.total,pstate --format=csv

### 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, shutil, zipfile
DRIVE = pathlib.Path('/content/drive/MyDrive/greenvl_t4')
BUNDLE = DRIVE / 'greenvl_t4_bundle.zip'
assert BUNDLE.exists(), f'Upload greenvl_t4_bundle.zip to {DRIVE} first'
print(BUNDLE, f'{BUNDLE.stat().st_size / 2**20:.0f} MiB')

### 3. Unpack (code and images on the Colab disk, results on Drive)

In [ ]:
WORK = pathlib.Path('/content/greenvl')
if not (WORK / 'Implementation').exists():
    zipfile.ZipFile(BUNDLE).extractall(WORK)
RESULTS = DRIVE / 'results'
if not RESULTS.exists():
    shutil.copytree(WORK / 'Implementation' / 'results', RESULTS)
os.environ['GREENVL_DATA'] = str(WORK / 'Datasets')
os.environ['GREENVL_RESULTS'] = str(RESULTS)
os.environ['PYTHONUNBUFFERED'] = '1'
os.chdir(WORK / 'Implementation')
print('code', os.getcwd()); print('data', os.environ['GREENVL_DATA']); print('results', RESULTS)

### 4. Libraries (same versions as on the Mac, except PyTorch, which stays Colab's CUDA build)

In [ ]:
!pip install -q transformers==5.17.0 peft==0.21.0 accelerate==1.15.0 bitsandbytes==0.50.2 nvidia-ml-py psutil
# Colab ships torchao 0.10, which peft 0.21 rejects as too old even though it is not used here: remove it
!pip uninstall -y -q torchao
!python -c "import torch, transformers, bitsandbytes, peft; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| transformers', transformers.__version__, '| bitsandbytes', bitsandbytes.__version__, '| peft', peft.__version__)"

### 5. Checks: energy counter and INT8 / NF4 layers on this GPU (~20 s)

In [ ]:
%%bash
python - <<'EOF'
import time, torch
from greenvl.energy import EnergyMeter
m = EnergyMeter('cuda')
assert m.available, m.error
idle = m.measure_idle(5)
print(f"idle: {idle['watts']:.1f} W over {idle['seconds']:.1f} s")
x = torch.randn(4096, 4096, device='cuda')
m.begin('load'); t = time.time()
while time.time() - t < 5:
    y = x @ x
torch.cuda.synchronize(); e = m.end('load')
print(f"matmul load: {e['total_j'] / e['seconds']:.1f} W")
import bitsandbytes as bnb
for name, layer in (('int8', bnb.nn.Linear8bitLt(768, 768, has_fp16_weights=False, threshold=6.0)),
                    ('nf4', bnb.nn.Linear4bit(768, 768, compute_dtype=torch.float16, quant_type='nf4'))):
    out = layer.cuda()(torch.randn(1, 768, device='cuda', dtype=torch.float16))
    print(name, 'layer OK', tuple(out.shape), out.dtype)
EOF

### 6. Smoke test (8 items, 1 repetition; ~10-15 min, mostly the per-configuration memory checks)

Checks the whole run end to end. Its results go to a separate `_smoke_t4` folder, deleted at the end of the cell.

In [ ]:
!python scripts/10_measure_variants.py --n 8 --repeats 1 --pause 1 --idle-seconds 5 --warmup 2 --label _smoke_t4
smoke = RESULTS / 'variants' / 'coco_ViT-L-14_lora8_lr1e-3_s0' / '_smoke_t4'
print('energy.md written:', (smoke / 'energy.md').exists())
shutil.rmtree(smoke, ignore_errors=True)

### 7. The measurement (3 repetitions; roughly 2-4 h on a T4)

Prints one line per window. If Colab disconnects, run cells 1-4 again and then this cell: finished windows are
skipped. `--repeats 5` later adds two more repetitions.

In [ ]:
!python scripts/10_measure_variants.py --label epoch_02_t4

### 8. Results

Everything is in `My Drive/greenvl_t4/results/variants/coco_ViT-L-14_lora8_lr1e-3_s0/epoch_02_t4/`
(`energy.md`, `energy.json`, `energy.jsonl`, `memory.json`). Copy that folder to the Mac as
`Implementation/results/variants/coco_ViT-L-14_lora8_lr1e-3_s0/epoch_02_t4/`, then:

    python scripts/11_analysis.py --platform t4 --platform-name "Colab T4" --parts frontier batch \
        --energy results/variants/coco_ViT-L-14_lora8_lr1e-3_s0/epoch_02_t4/energy.json

In [ ]:
print((RESULTS / 'variants' / 'coco_ViT-L-14_lora8_lr1e-3_s0' / 'epoch_02_t4' / 'energy.md').read_text())